# 1 · Fundamentos de LLMs: de tokens a una primera aplicación

Prerrequisitos: Python y fundamentos de ML.

Explora tokens, representación, generación y contexto. Después completa `build_messages`, cambia un precio y comprueba qué ocurre cuando falta un atributo. Al terminar podrás explicar qué calcula un LLM, qué cambia durante el entrenamiento y qué aporta la aplicación que lo utiliza.

## Qué vamos a observar
¿Cómo puede un sistema producir una frase nueva sin tener esa frase guardada como respuesta?
Conserva esta pregunta. Hoy observaremos representación, predicción, generación y contexto;
construiremos una llamada pequeña solo cuando esos conceptos tengan sentido.
La celda de entorno está suministrada: no necesitas crear una base de datos ni instalar un modelo.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv

if not os.getenv("CARRITO_NOTEBOOK_CHECK"):
    load_dotenv(ROOT / ".env")
from carrito.foundations import (
    conditional_probabilities,
    greedy_demo,
    load_tokenization_examples,
    softmax,
)

RUN_LIVE = False  # Activar explícitamente permite llamadas de pago.
COMPLETION = {}
def check(name, condition):
    COMPLETION[name] = bool(condition)
    print(("OK" if condition else "PENDIENTE") + ": " + name)
print("Sin inferencia de LLM: tokenizer medido + toy matemático + respuestas fixture etiquetadas.")

## Una historia de representaciones y objetivos
Un modelo de lenguaje asigna probabilidades a continuaciones. Los modelos de conteo
estiman frecuencias de secuencias cortas; las redes neuronales aprenden representaciones
y relaciones; el Transformer permite combinar posiciones mediante atención.
Escalar datos y entrenamiento mejora capacidades, y la adaptación a instrucciones cambia
cómo se usa esa capacidad. Un chat añade roles e historial a un modelo: la interfaz no es el modelo.

**Pregunta:** ¿memorizar una frase, aprender una regularidad y consultar un dato actual
son la misma operación? Mantén separadas capacidad aprendida, evidencia del contexto y acceso a tools.

## Demo 1: un token no es necesariamente una palabra
Estos IDs y fragmentos se midieron con **tiktoken / cl100k_base** y están guardados en
`data/foundations/tokenization.json`. No son una separación inventada a mano ni una llamada
a un modelo. El tokenizer del `OPENAI_MODEL` elegido puede ser distinto.
Los IDs son índices de vocabulario; no expresan significado ni cercanía semántica.
Predice cuántos tokens habrá antes de ejecutar. Compara palabra, espacio, cifra y puntuación.

In [ ]:
measured = load_tokenization_examples()
print("Tokenizer:", measured["encoding"], "· tiktoken", measured["library_version"])
for row in measured["examples"]:
    print(repr(row["text"]))
    print("  IDs:", row["token_ids"])
    print("  piezas:", [piece["display"] for piece in row["pieces"]])
    print("  palabras separadas por espacios:", len(row["text"].split()), "tokens:", row["token_count"])
    recovered = b"".join(bytes.fromhex(piece["bytes_hex"]) for piece in row["pieces"]).decode()
    assert recovered == row["text"]

## Demo 2: distribución del siguiente token, condicionada al prefijo
La expresión `P(siguiente token | tokens anteriores)` devuelve una distribución sobre un vocabulario.
**Toy de ocho palabras:** las puntuaciones de abajo están escritas a mano; no son probabilidades
medidas de un LLM ni un Transformer entrenado. El vocabulario del toy usa palabras enteras para
poder leer los números. Es distinto del tokenizer real de la demo anterior.

El prefijo cambia la distribución aunque la tabla del toy permanezca fija.
Predice qué gana después de «El pedido llega» y después de «El pedido sale».

In [ ]:
for prefix in ["El pedido llega", "El pedido sale"]:
    distribution = conditional_probabilities(prefix)
    print("P(siguiente |", repr(prefix), ")")
    print({token: round(probability, 4) for token, probability in distribution.items()})
    print("Greedy elegiría:", max(distribution, key=distribution.get))
    assert abs(sum(distribution.values()) - 1) < 1e-12
print("Generación autoregresiva: añadir un token y volver a calcular.")
for step in greedy_demo():
    print(repr(step["prefix"]), "→", step["next_token"])

## Transformer: de IDs a representaciones que usan contexto
Recorrido conceptual: IDs → embeddings y posición → bloques de atención causal y MLP
(con conexiones residuales y normalización) → logits sobre el vocabulario → siguiente token.
El embedding es un vector aprendido; un ID por sí solo no es ese vector.

**Miniatura de una sola cabeza:** fijamos tres puntuaciones de atención y tres valores escalares.
La fila `i` solo puede mezclar posiciones `0..i`; las posteriores quedan en cero por la máscara causal.
No implementamos Q/K/V aprendidos, proyecciones, múltiples cabezas ni bloques completos.
Una matriz de atención no es una prueba de verdad ni una explicación completa del modelo.

In [ ]:
tokens = ["El", "pedido", "llega"]
values = [10.0, 20.0, 30.0]  # Valores escalares ilustrativos, no embeddings reales.
scores_by_position = [[2.0], [1.0, 2.0], [0.0, 1.0, 2.0]]
for position, scores in enumerate(scores_by_position):
    allowed_weights = softmax(scores)
    causal_row = allowed_weights + [0.0] * (len(tokens) - position - 1)
    mixed_value = sum(weight * value for weight, value in zip(causal_row, values, strict=True))
    print(tokens[position], "pesos=", [round(w, 3) for w in causal_row], "mezcla=", round(mixed_value, 3))
    assert all(weight == 0 for weight in causal_row[position + 1:])

## Qué se aprende y qué permanece fijo al preguntar
En pretraining, el propio texto aporta objetivos del siguiente token: es aprendizaje autosupervisado.
La pérdida compara probabilidades con el token objetivo; el optimizador actualiza parámetros.
La adaptación a instrucciones y preferencias usa objetivos/datos adicionales para cambiar comportamiento.
Ninguna de estas operaciones equivale a pegar un catálogo en el prompt.

En inferencia ordinaria, los parámetros permanecen fijos y cambian entradas, activaciones y salida.
La ventana de contexto tiene un límite; recibir información en ella no demuestra que se haya guardado
en los pesos ni garantiza recordarla en otra conversación. Observa el desplazamiento input/target:

In [ ]:
from math import log

training_tokens = ["El", "pedido", "llega", "mañana"]
training_pairs = [(training_tokens[:i], training_tokens[i]) for i in range(1, len(training_tokens))]
print("Ejemplos prefix → objetivo:", training_pairs)
print("Pérdida ilustrativa -log(p_objetivo): p=0.2 →", round(-log(0.2), 3), "; p=0.8 →", round(-log(0.8), 3))
print("Esta celda no entrena: calcula objetivos y pérdidas para explicarlos.")

## Inferencia y decoding: temperatura sobre logits fijos
`p_i = exp(z_i / T) / sum(exp(z_j / T))`, con `T > 0`.
Comparamos **los mismos logits ilustrativos** `[2, 1, 0]` con tres temperaturas,
normalizados solo entre los tres candidatos «mañana», «hoy», «tarde». Es un vocabulario
reducido para ver las cuentas; no predice la fecha de entrega de ningún pedido.
Menor T concentra masa; mayor T la reparte. El orden de los logits no cambia.
Greedy elige el máximo; sampling extrae según la distribución. Nuestro cálculo no muestrea.
No uses temperatura como «control de alucinación»: una distribución muy concentrada también
puede favorecer una continuación falsa. No todos los modelos/API exponen este parámetro.

In [ ]:
fixed_logits = [2.0, 1.0, 0.0]
for temperature in [0.5, 1.0, 2.0]:
    probabilities = softmax(fixed_logits, temperature)
    print("T=", temperature, "P=", [round(p, 4) for p in probabilities], "suma=", round(sum(probabilities), 6))
print("Ningún parámetro aprendido se ha actualizado; solo transformamos puntuaciones fijas.")

In [ ]:
PRODUCTS = [
    {"id": "P001", "name": "Auriculares Nube", "price_eur": 69.90},
    {"id": "P002", "name": "Auriculares Estudio", "price_eur": 129.00},
    {"id": "P003", "name": "Altavoz Brisa", "price_eur": 45.90},
    {"id": "P004", "name": "Micrófono Claro", "price_eur": 59.90},
    {"id": "P005", "name": "Teclado Arena", "price_eur": 49.90},
]
# Cinco fichas abreviadas del catálogo real: sin atributos técnicos todavía.
# Los atributos completos y los veinte productos se consultan desde S3.
question = "Busco auriculares inalámbricos por menos de 80 euros."

## Contexto contrafactual: el mismo modelo, otro dato disponible
Las cinco fichas anteriores son evidencia, no conocimientos guardados en los pesos.
Cambiamos solo P001 de 69,90 a 89,90 €, manteniendo pregunta e instrucción.
Con presupuesto de 80 €, la respuesta sustentada debería cambiar.
Las respuestas offline de esta demo están **escritas de antemano**: muestran la comparación
esperada, no prueban que un modelo siga la evidencia. En live se conservan modelo/configuración
y se observan las salidas reales; este código nunca entrena.

In [ ]:
import inspect
from copy import deepcopy

from carrito.model import native_call

context_b = deepcopy(PRODUCTS)
context_b[0]["price_eur"] = 89.90
demo_question = "¿P001 cabe en mi presupuesto de 80 euros?"
fixed_instruction = "Usa solo las fichas dadas. Si falta un atributo, reconoce que no lo sabes."
def prepared_messages(products):
    return [{"role": "system", "content": fixed_instruction},
            {"role": "user", "content": demo_question + "\nCATÁLOGO: " + json.dumps(products, ensure_ascii=False)}]
context_inputs = [prepared_messages(PRODUCTS), prepared_messages(context_b)]
authored_outputs = ["[FIXTURE] Sí: 69,90 € está dentro de 80 €.", "[FIXTURE] No: 89,90 € supera 80 €."]
for label, payload, expected in zip(["A", "B"], context_inputs, authored_outputs, strict=True):
    print("Contexto", label, json.dumps(payload, ensure_ascii=False))
    observed = native_call(payload) if RUN_LIVE else {"mode": "authored_fixture", "output_text": expected, "usage": None}
    print("Salida:", observed)
assert PRODUCTS[0]["price_eur"] == 69.90  # El contexto original no se ha mutado.
print("Llamada SDK suministrada, sin tools ni loop:")
print(inspect.getsource(native_call))

En la función anterior localiza `model`, `input`, `max_output_tokens` y `store=False`.
En la respuesta distingue `output_text`, los items `output`, `status` y `usage`.
`usage=None` en nuestras fixtures significa «no medido», nunca cero tokens de un modelo real.
Si una llamada real falla por red, cuota o acceso, el error no demuestra incapacidad lingüística.
Una respuesta incompleta o un rechazo tampoco debe presentarse como éxito.

## Capacidades, límites y entrada multimodal
Generar, resumir y extraer patrones no garantiza disponer de hechos actuales, respetar un límite
o reconocer información ausente. En las fichas abreviadas no figura resistencia al agua ni conectividad.
La imagen amplía el tipo de entrada; no convierte una ilustración en prueba de stock o garantía.
Mira la tarjeta antes de ejecutar la demo preparada: ¿qué puede observarse y qué habría que consultar?

In [ ]:
import subprocess

from IPython.display import Image, display

display(Image(filename=str(ROOT / "data/demo/product-card.png"), width=480))
demo = subprocess.run([sys.executable, str(ROOT / "examples/multimodal.py"),
                       "--mode", "live" if RUN_LIVE else "fixture"],
                      cwd=ROOT, capture_output=True, text=True, check=True)
print(demo.stdout)

## Modelo, aplicación, workflow y agente
- **Modelo:** transforma un contexto en probabilidades y contenido generado.
- **Aplicación:** añade interfaz, instrucciones, datos, validaciones y presentación.
- **Workflow:** el software fija una secuencia; por ejemplo, leer datos → llamar → comprobar.
- **Agente:** el modelo puede elegir el siguiente paso mediante tools, dentro de límites del host.

Nuestro primer lab es una aplicación con una llamada. No permite que el modelo consulte SQLite,
elija tools ni escriba devoluciones. En S3–S4 veremos qué componentes adicionales hacen falta.
Predice quién debe comprobar un precio: ¿basta una instrucción al modelo o necesitamos evidencia y checks?

## Primera aplicación: una función y dos contrastes

1. Completa `build_messages` con dos roles, las cinco fichas y una regla ante datos ausentes.
2. Ejecuta el caso base y explica qué puede afirmarse sobre P001.
3. Cambia solo su precio en una copia del contexto y compara contra el presupuesto de 80 €.
4. Pregunta por resistencia al agua y anota qué información falta.

Los contrastes y la llamada están suministrados. Pista: usa `json.dumps(products, ensure_ascii=False)` en el mensaje de usuario. El notebook puede ejecutarse sin completar la función: los checks quedan PENDIENTE.

In [ ]:
def build_messages(question, products):
    # TODO: separar instrucciones de petición/evidencia y reconocer datos ausentes.
    return []
messages = build_messages(question, PRODUCTS)

In [ ]:
system_text = messages[0].get("content", "") if messages else ""
user_text = messages[1].get("content", "") if len(messages) > 1 else ""
check("dos roles separados", len(messages) == 2 and [m["role"] for m in messages] == ["system", "user"])
check("cinco productos en contexto", all(p["id"] in user_text for p in PRODUCTS))
check("regla para información ausente", any(word in system_text.lower() for word in ["falta", "ausente", "desconoc", "no sabes"]))
print(json.dumps(messages, ensure_ascii=False, indent=2))

In [ ]:
changed_products = deepcopy(PRODUCTS)
changed_products[0]["price_eur"] = 89.90  # El único dato que modificamos.
lab_cases = [
    ("base", "¿P001 cabe en 80 euros?", PRODUCTS, "P001 cuesta 69,90 €; sí cabe."),
    ("precio cambiado", "¿P001 cabe en 80 euros?", changed_products, "Ahora la ficha dice 89,90 €; no cabe."),
    ("dato ausente", "¿P001 es resistente al agua?", PRODUCTS, "La ficha no aporta resistencia al agua."),
]
lab_observations = []
for label, prompt, evidence, expected in lab_cases:
    payload = build_messages(prompt, evidence)
    if RUN_LIVE and len(payload) == 2:
        observed = native_call(payload)
    else:
        observed = {"mode": "expectation_only", "output_text": None, "usage": None}
    lab_observations.append({"case": label, "messages": payload, "observed": observed,
                             "expected": expected, "my_explanation": ""})
print(json.dumps(lab_observations, ensure_ascii=False, indent=2))
print("Sin API, completa tu explicación de la evidencia; no registres la expectativa como salida real.")

## Primera evaluación: cinco casos

Registra lo observado en cada caso. Sin API, etiqueta el análisis como fixture y distingue salida preparada, expectativa y explicación.

| Caso | Qué comprobar |
|---|---|
| ¿P001 cabe en 80 €? Contexto A | Precio de 69,90 € sustentado |
| Misma pregunta, contexto B | Precio de 89,90 €: cambia la conclusión |
| ¿P001 es resistente al agua? | El atributo no existe en las fichas |
| ¿Qué compro? | Faltan preferencias |
| Monitor por 20 € | No inventar producto ni descuento |

In [ ]:
manual_cases = ["¿P001 cabe en 80 €? (contexto A)", "¿P001 cabe en 80 €? (contexto B)", "¿P001 es resistente al agua?", "¿Qué compro?", "Monitor por 20 €"]
# Salidas escritas para discusión. La tercera contiene un fallo deliberado.
observed_fixtures = ["P001 cuesta 69,90 € y cabe en 80 €.", "P001 cuesta 89,90 €; no cabe en 80 €.",
                     "P001 es resistente al agua.", "¿Qué necesitas y con qué presupuesto?",
                     "No hay monitores en las cinco fichas."]
observations = [{"question": q, "work": "tres propios" if i < 3 else "dos guiados",
                 "mode": "authored_fixture", "observed": answer, "my_explanation": ""}
                for i, (q, answer) in enumerate(zip(manual_cases, observed_fixtures, strict=True))]
print(json.dumps(observations, ensure_ascii=False, indent=2))

In [ ]:
# Completa my_explanation en las tres observaciones propias; anota las dos conclusiones guiadas.

**Aceptación técnica:** pasan los tres checks de `build_messages`. **Aceptación conceptual:**
explicas token frente a palabra, generación paso a paso, parámetros frente a contexto y por qué
cambiar T no demuestra corrección. Entregas los tres contrastes del lab y las cinco notas de cierre
sobre los cinco casos, con el fallo de atributo inventado identificado.
Los checks no sustituyen esa explicación ni miden la calidad de un LLM real.

**Checkpoint siguiente:** S2 trae su catálogo y mensajes preparados; no depende de completar S1.
Fuentes: [tiktoken oficial](https://github.com/openai/tiktoken) y
[generación de texto en la API](https://developers.openai.com/api/docs/guides/text).
Las cifras del toy y de atención son ilustrativas; los IDs del tokenizer son mediciones guardadas.

In [ ]:
print(json.dumps(COMPLETION, ensure_ascii=False, indent=2))
print("CHECKPOINT_COMPLETO" if all(COMPLETION.values()) else "Completa las celdas TODO y repite los checks.")